In [2]:
import json
import pandas as pd
from tqdm import tqdm
import re
import numpy as np
import matplotlib.pyplot as plt
import math

## Load Processed FDA file

In [3]:
out_dir = "./out"

In [99]:
# produced in FDA_01
df_orig = pd.read_csv(f"{out_dir}/FDA_full_AP_mapped_active_ingredients_with_parent.csv")
df_orig.shape

(49990, 27)

In [100]:
df_orig_info = df_orig[['sponsor_name','year','application_number', 'brand_name', 'active_ingredients_split', 'drug_umls_term_norm', 'clinical_studies', 'indications_first_sent', 'indications_and_usage']]
df_orig_info.head()

,sponsor_name,year,application_number,brand_name,active_ingredients_split,drug_umls_term_norm,clinical_studies,indications_first_sent,indications_and_usage
0,WATSON LABS,2002,ANDA076194,LISINOPRIL AND HYDROCHLOROTHIAZIDE,HYDROCHLOROTHIAZIDE,Hydrochlorothiazide,[],INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...
1,WATSON LABS,2002,ANDA076194,LISINOPRIL AND HYDROCHLOROTHIAZIDE,LISINOPRIL,Lisinopril,[],INDICATIONS AND USAGE Lisinopril and Hydrochlo...,INDICATIONS AND USAGE Lisinopril and Hydrochlo...
2,ROCKWELL MEDCL,2003,ANDA076206,CALCITRIOL,CALCITRIOL,Calcitriol,NaN,NaN,NaN
3,APOTEX,2004,ANDA076212,CARBIDOPA AND LEVODOPA,CARBIDOPA,Carbidopa,NaN,NaN,NaN
4,APOTEX,2004,ANDA076212,CARBIDOPA AND LEVODOPA,LEVODOPA,Levodopa,NaN,NaN,NaN


## Load Preclinical DS drugs

In [103]:
terms_path = "./data/unique_drug_terms_292514.csv" # from Translation_01_Drug, preclinical pipeline
terms = pd.read_csv(terms_path)

In [104]:
terms_for_details = terms[terms["merged_umls_label"].str.len() > 2]
terms_for_details = terms_for_details[terms_for_details['n_articles']>1]

terms_for_details.head()

,merged_umls_label,n_articles
0,Dexamethasone,5806
1,Acetylcysteine,4559
2,NG-Nitroarginine Methyl Ester,4306
3,Sirolimus,4107
4,Doxorubicin,4021


In [105]:
terms_for_details.shape

(74547, 2)

In [106]:
terms_for_details[terms_for_details["merged_umls_label"].str.contains("zoloft", case=False, na=False)].head()

,merged_umls_label,n_articles


## Merge drugs from dataset with FDA details

In [107]:
# 1) Build normalized join keys (upper + strip)
df_counts = terms_for_details.copy()
df_exploded = df_orig_info.copy()

df_counts["__key"] = df_counts["merged_umls_label"].astype(str).str.upper().str.strip()
df_exploded["__key"] = df_exploded["drug_umls_term_norm"].astype(str).str.upper().str.strip()

# 2) Left-merge ON the ingredient (keep all rows from df_exploded)
df_merged = df_counts.merge(
    df_exploded,
    on="__key",
    how="left"
)

# 3) Clean up
df_merged = df_merged.drop(columns="__key")
df_merged.head()

,merged_umls_label,n_articles,sponsor_name,year,application_number,brand_name,active_ingredients_split,drug_umls_term_norm,clinical_studies,indications_first_sent,indications_and_usage
0,Dexamethasone,5806,SUN PHARM INDUSTRIES,1974.0,ANDA084013,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN
1,Dexamethasone,5806,WHITEWORTH TOWN PLSN,1975.0,ANDA084327,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN
2,Dexamethasone,5806,WATSON LABS,1977.0,ANDA085455,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN
3,Dexamethasone,5806,HIKMA,1983.0,ANDA088248,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,[],INDICATIONS AND USAGE Allergic States Control ...,INDICATIONS AND USAGE Allergic States Control ...
4,Dexamethasone,5806,EYEPOINT PHARMS,2018.0,NDA208912,DEXYCU KIT,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN


In [108]:
df_ds_drugs_with_FDA_info = df_merged[df_merged["application_number"].notna()]
df_ds_drugs_with_FDA_info.head()

,merged_umls_label,n_articles,sponsor_name,year,application_number,brand_name,active_ingredients_split,drug_umls_term_norm,clinical_studies,indications_first_sent,indications_and_usage
0,Dexamethasone,5806,SUN PHARM INDUSTRIES,1974.0,ANDA084013,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN
1,Dexamethasone,5806,WHITEWORTH TOWN PLSN,1975.0,ANDA084327,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN
2,Dexamethasone,5806,WATSON LABS,1977.0,ANDA085455,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN
3,Dexamethasone,5806,HIKMA,1983.0,ANDA088248,DEXAMETHASONE,DEXAMETHASONE,Dexamethasone,[],INDICATIONS AND USAGE Allergic States Control ...,INDICATIONS AND USAGE Allergic States Control ...
4,Dexamethasone,5806,EYEPOINT PHARMS,2018.0,NDA208912,DEXYCU KIT,DEXAMETHASONE,Dexamethasone,NaN,NaN,NaN


In [109]:
df_ds_drugs_with_FDA_info.merged_umls_label.nunique()

2745

In [110]:
df_ds_drugs_with_FDA_info.to_csv(f"{out_dir}/df_ds_drugs_with_FDA_info.csv", index=False)

In [111]:
df_with_indications = df_ds_drugs_with_FDA_info[df_ds_drugs_with_FDA_info["indications_first_sent"].notna()]
df_with_indications['unique_id'] = (
    df_with_indications['merged_umls_label']
    .str.replace(" ", "_")
    .str.lower()
    +"_" + df_with_indications['application_number'].astype(str)
)


/tmp/ipykernel_2820109/2705641560.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_with_indications['unique_id'] = (


In [113]:
df_with_indications[df_with_indications["merged_umls_label"].str.contains("cariprazine", case=False, na=False)]

,merged_umls_label,n_articles,sponsor_name,year,application_number,brand_name,active_ingredients_split,drug_umls_term_norm,clinical_studies,indications_first_sent,indications_and_usage,unique_id
28150,Cariprazine,35,ABBVIE,2015.0,NDA204370,VRAYLAR,CARIPRAZINE,Cariprazine,[],1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,cariprazine_NDA204370
28151,Cariprazine,35,ABBVIE,2015.0,NDA204370,VRAYLAR,CARIPRAZINE HYDROCHLORIDE,Cariprazine,[],1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,1 INDICATIONS AND USAGE VRAYLAR ® is indicated...,cariprazine_NDA204370


In [114]:
df_with_indications.shape

(19901, 12)

In [115]:
df_with_indications.merged_umls_label.nunique()

2201

### save for LLM processing

In [116]:
df_with_indications.to_csv(f"{out_dir}/drugs_with_indications.csv", index=False)

In [49]:
# Calculate chunk size
num_chunks = 3
chunk_size = math.ceil(len(df_with_indications) / num_chunks)

# Split and save each chunk
for i in range(num_chunks):
    start_idx = i * chunk_size
    end_idx = start_idx + chunk_size
    chunk_df = df_with_indications.iloc[start_idx:end_idx]
    print(f"chunk size {chunk_df.shape}")
    chunk_df.to_csv(f"{out_dir}/drugs_with_indications_chunk_{i+1}.csv", index=False)


chunk size (4255, 12)
chunk size (4255, 12)
chunk size (4254, 12)


In [50]:
df_ds_drugs_with_FDA_info.merged_umls_label.nunique()

1673